In [3]:
import pyomo.environ as pyo
import pandas as pd
import numpy as np
import math

## Simulate the Williams-Otto process model

In [11]:
model = pyo.ConcreteModel()

# define the model parameters
rho = 50
a1 = 5.9755 * 10**9
a2 = 2.5962 * 10**12
a3 = 9.6283 * 10**15
reaction_number = [1, 2, 3]

# define the model parameters
model.alpha_1 = pyo.Var(bounds=(0, None), initialize=1)
model.alpha_1.fix(pyo.log(a1))

model.alpha_2 = pyo.Var(bounds=(0, None), initialize=1)
model.alpha_2.fix(pyo.log(a2))

model.alpha_3 = pyo.Var(bounds=(0, None), initialize=1)
model.alpha_3.fix(pyo.log(a3))

model.E1 = pyo.Var(bounds=(0, None), initialize=10)
model.E1.fix(120)

model.E2 = pyo.Var(bounds=(0, None), initialize=10)
model.E2.fix(150)

model.E3 = pyo.Var(bounds=(0, None), initialize=10)
model.E3.fix(200)

# define the model variables
# decision variables
model.V = pyo.Var(bounds=(0.03, 0.1), initialize=0.06)
model.T = pyo.Var(bounds=(5.8, 6.8), initialize=5.8)
model.T_reparam = pyo.Var(bounds=(1/6.8, 1/5.8),)
model.FA = pyo.Var(bounds=(1, None), initialize=10)
model.FB = pyo.Var(bounds=(1, None), initialize=20)
model.eta = pyo.Var(bounds=(0, 1), initialize=0.1)

# define the components
components = ['A','B','C','E','P','G']

# add the effluent stream of the reactor
model.Feff = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':5,'G':1})
model.Feff_sum = pyo.Var(within=pyo.PositiveReals, initialize=52)

# add the product, waste, purge, and recycle streams
model.FP = pyo.Var(bounds=(0, 4.763), initialize=0.5)
model.FG = pyo.Var(within=pyo.NonNegativeReals, initialize=1)
model.Fpurge = pyo.Var(within=pyo.NonNegativeReals, initialize=0)
model.FR = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':0.3,'G':0})

# add the mass fractions
model.X = pyo.Var(components, bounds=(0, 1), initialize={'A':0.2,'B':0.2,'C':0.2,'E':0.2,'P':0.2,'G':0.2})

# add the temperature constraints
model.temp_rule = pyo.Constraint(
    expr = model.T_reparam == 1 / model.T
)

# add the rate constants
model.k = pyo.Var(reaction_number, bounds=(0, None), initialize={1:6.18, 2:15.2, 3:10.2})
model.k_reparam = pyo.Var(reaction_number, bounds=(0, None),)

# add the reaction rates
model.r = pyo.Var(reaction_number, bounds=(0, None))

# calculate the reparameterized rate constants
def k_reparam_rule(model, i):
    if i == 1:
        return model.k_reparam[i] == model.alpha_1 - model.E1 * model.T_reparam
    elif i == 2:
        return model.k_reparam[i] == model.alpha_2 - model.E2 * model.T_reparam
    else:
        return model.k_reparam[i] == model.alpha_3 - model.E3 * model.T_reparam
        
model.k_reparam_eq = pyo.Constraint(
    reaction_number, rule=k_reparam_rule
)

# calculate the original rate constants
def k_rule(model, i):
    return model.k[i] == pyo.exp(model.k_reparam[i])

model.k_eq = pyo.Constraint(
    reaction_number, rule=k_rule
)

# define the rates of reactions
def reaction_rate_rule(model, i):
    if i == 1:
        return model.r[i] == model.k[i] * model.X['A'] * model.X['B'] * model.V * rho
    elif i == 2:
        return model.r[i] == model.k[i] * model.X['C'] * model.X['B'] * model.V * rho
    else:
        return model.r[i] == model.k[i] * model.X['P'] * model.X['C'] * model.V * rho

model.reaction_rate = pyo.Constraint(
    reaction_number, rule=reaction_rate_rule
)

# define the species mass balances
model.A_bal = pyo.Constraint(
    expr = model.Feff['A'] == model.FA + model.FR['A'] - model.r[1]
)

model.B_bal = pyo.Constraint(
    expr = model.Feff['B'] == model.FB + model.FR['B'] - (model.r[1] + model.r[2])
)

model.C_bal = pyo.Constraint(
    expr = model.Feff['C'] == model.FR['C'] + 2 * model.r[1] - 2 * model.r[2] - model.r[3]
)

model.E_bal = pyo.Constraint(
    expr = model.Feff['E'] == model.FR['E'] + 2 * model.r[2]
)

model.P_bal = pyo.Constraint(
    expr = model.Feff['P'] == 0.1 * model.FR['E'] + model.r[2] - 0.5 * model.r[3]
)

model.G_bal = pyo.Constraint(
    expr = model.Feff['G'] == 1.5 * model.r[3]
)

# calculate the total effluent flow
model.total_effluent = pyo.Constraint(
    expr = model.Feff_sum == sum(model.Feff[j] for j in components)
)

# add the constraints on the total effluent flow
model.total_effluent_constraint = pyo.Constraint(
    expr = model.Feff_sum >= 1
)

# ensure mass fraction balance
def mole_frac_rule(model, j):
    return model.Feff[j] == model.Feff_sum * model.X[j]

model.mole_frac = pyo.Constraint(components, rule=mole_frac_rule)

# calculate the product flowrate 
model.product = pyo.Constraint(
    expr = model.FP == model.Feff['P'] - 0.1 * model.Feff['E']
)

# calculate the flowrate of the waste
model.waste = pyo.Constraint(
    expr = model.FG == model.Feff['G']
)

# calculate the flowrate of the purge stream
model.purge = pyo.Constraint(
    expr = model.Fpurge == model.eta * (model.Feff['A'] + model.Feff['B'] + model.Feff['C'] + 1.1 * model.Feff['E'])
)

# calculate the flowrate of the recycle stream
def recycle_rule(model, j):
    if j == "G":
        return model.FR[j] == 0
    elif j == "P":
        return model.FR[j] == (1 - model.eta) * 0.1 * model.Feff["E"]
    else:
        return model.FR[j] == (1 - model.eta) * model.Feff[j]

model.recycle = pyo.Constraint(components, rule=recycle_rule)

# define the objective function
model.obj = pyo.Objective(
    expr = 100 * (2207*model.FP + 50*model.Fpurge - 168*model.FA - 252*model.FB - 2.22*model.Feff_sum 
                  - 84*model.FG - 60*model.V*rho)/(600 * model.V * rho),
    sense=pyo.maximize
)

# define the solver
solver = pyo.SolverFactory('ipopt')

# solve the model
results = solver.solve(model, tee=True)

# print the results
print("ROI =", pyo.value(model.obj))
for v in model.component_objects(pyo.Var, active=True):
    print("\n",v)
    for index in v:
        print(index, pyo.value(v[index]))

Ipopt 3.14.19: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.19, running with linear solver MUMPS 5.8.2.

Number of nonzeros in equality constraint Jacobian...:      104
Number of nonzeros in inequality constraint Jacobian.:        1
Number of nonzeros in Lagrangian Hessian.............:       37

Total number of variables............................:       37
                     variables with only lower bounds:       26
                variables with lower and upper bounds:       11
                     variables with only upper bounds:        0
Total number of equality constraints.................:       32
Total number

In [13]:
def williams_otto_plant():
    model = pyo.ConcreteModel()

    # define the model parameters
    rho = 50
    a1 = 5.9755 * 10**9
    a2 = 2.5962 * 10**12
    a3 = 9.6283 * 10**15
    reaction_number = [1, 2, 3]
    
    # define the model parameters
    model.alpha_1 = pyo.Var(bounds=(0, None), initialize=1)
    model.alpha_1.fix(pyo.log(a1))
    
    model.alpha_2 = pyo.Var(bounds=(0, None), initialize=1)
    model.alpha_2.fix(pyo.log(a2))
    
    model.alpha_3 = pyo.Var(bounds=(0, None), initialize=1)
    model.alpha_3.fix(pyo.log(a3))
    
    model.E1 = pyo.Var(bounds=(0, None), initialize=10)
    model.E1.fix(120)
    
    model.E2 = pyo.Var(bounds=(0, None), initialize=10)
    model.E2.fix(150)
    
    model.E3 = pyo.Var(bounds=(0, None), initialize=10)
    model.E3.fix(200)

    # define the decision variables
    model.V = pyo.Var(bounds=(0.03, 0.1), initialize=0.06)
    model.T = pyo.Var(bounds=(5.8, 6.8), initialize=5.8)
    model.T_reparam = pyo.Var(bounds=(1/6.8, 1/5.8),)
    model.FA = pyo.Var(bounds=(1, None), initialize=10)
    model.FB = pyo.Var(bounds=(1, None), initialize=20)
    model.eta = pyo.Var(bounds=(0, 1), initialize=0.1)
    
    # define the components
    components = ['A','B','C','E','P','G']
    
    # add the effluent stream of the reactor
    model.Feff = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':5,'G':1})
    model.Feff_sum = pyo.Var(within=pyo.PositiveReals, initialize=52)
    
    # add the product, waste, purge, and recycle streams
    model.FP = pyo.Var(bounds=(0, 4.763), initialize=0.5)
    model.FG = pyo.Var(within=pyo.NonNegativeReals, initialize=1)
    model.Fpurge = pyo.Var(within=pyo.NonNegativeReals, initialize=0)
    model.FR = pyo.Var(components, within=pyo.NonNegativeReals, initialize={'A':10,'B':30,'C':3,'E':3,'P':0.3,'G':0})
    
    # add the mass fractions
    model.X = pyo.Var(components, bounds=(0, 1), initialize={'A':0.2,'B':0.2,'C':0.2,'E':0.2,'P':0.2,'G':0.2})
    
    # add the temperature constraints
    model.temp_rule = pyo.Constraint(
        expr = model.T_reparam == 1 / model.T
    )
    
    # add the rate constants
    model.k = pyo.Var(reaction_number, bounds=(0, None), initialize={1:6.18, 2:15.2, 3:10.2})
    model.k_reparam = pyo.Var(reaction_number, bounds=(0, None),)
    
    # add the reaction rates
    model.r = pyo.Var(reaction_number, bounds=(0, None))
    
    # calculate the reparameterized rate constants
    def k_reparam_rule(model, i):
        if i == 1:
            return model.k_reparam[i] == model.alpha_1 - model.E1 * model.T_reparam
        elif i == 2:
            return model.k_reparam[i] == model.alpha_2 - model.E2 * model.T_reparam
        else:
            return model.k_reparam[i] == model.alpha_3 - model.E3 * model.T_reparam
            
    model.k_reparam_eq = pyo.Constraint(
        reaction_number, rule=k_reparam_rule
    )
    
    # calculate the original rate constants
    def k_rule(model, i):
        return model.k[i] == pyo.exp(model.k_reparam[i])
    
    model.k_eq = pyo.Constraint(
        reaction_number, rule=k_rule
    )

    # define the rates of reactions
    def reaction_rate_rule(model, i):
        if i == 1:
            return model.r[i] == model.k[i] * model.X['A'] * model.X['B'] * model.V * rho
        elif i == 2:
            return model.r[i] == model.k[i] * model.X['C'] * model.X['B'] * model.V * rho
        else:
            return model.r[i] == model.k[i] * model.X['P'] * model.X['C'] * model.V * rho
    
    model.reaction_rate = pyo.Constraint(
        reaction_number, rule=reaction_rate_rule
    )
    
    # define the species mass balances
    model.A_bal = pyo.Constraint(
        expr = model.Feff['A'] == model.FA + model.FR['A'] - model.r[1]
    )
    
    model.B_bal = pyo.Constraint(
        expr = model.Feff['B'] == model.FB + model.FR['B'] - (model.r[1] + model.r[2])
    )
    
    model.C_bal = pyo.Constraint(
        expr = model.Feff['C'] == model.FR['C'] + 2 * model.r[1] - 2 * model.r[2] - model.r[3]
    )
    
    model.E_bal = pyo.Constraint(
        expr = model.Feff['E'] == model.FR['E'] + 2 * model.r[2]
    )
    
    model.P_bal = pyo.Constraint(
        expr = model.Feff['P'] == 0.1 * model.FR['E'] + model.r[2] - 0.5 * model.r[3]
    )
    
    model.G_bal = pyo.Constraint(
        expr = model.Feff['G'] == 1.5 * model.r[3]
    )
    
    # calculate the total effluent flow
    model.total_effluent = pyo.Constraint(
        expr = model.Feff_sum == sum(model.Feff[j] for j in components)
    )
    
    # add the constraints on the total effluent flow
    model.total_effluent_constraint = pyo.Constraint(
        expr = model.Feff_sum >= 1
    )
    
    # ensure mass fraction balance
    def mole_frac_rule(model, j):
        return model.Feff[j] == model.Feff_sum * model.X[j]
    
    model.mole_frac = pyo.Constraint(components, rule=mole_frac_rule)
    
    # calculate the product flowrate 
    model.product = pyo.Constraint(
        expr = model.FP == model.Feff['P'] - 0.1 * model.Feff['E']
    )

    # calculate the flowrate of the waste
    model.waste = pyo.Constraint(
        expr = model.FG == model.Feff['G']
    )
    
    # calculate the flowrate of the purge stream
    model.purge = pyo.Constraint(
        expr = model.Fpurge == model.eta * (model.Feff['A'] + model.Feff['B'] + model.Feff['C'] + 1.1 * model.Feff['E'])
    )
    
    # calculate the flowrate of the recycle stream
    def recycle_rule(model, j):
        if j == "G":
            return model.FR[j] == 0
        elif j == "P":
            return model.FR[j] == (1 - model.eta) * 0.1 * model.Feff["E"]
        else:
            return model.FR[j] == (1 - model.eta) * model.Feff[j]
    
    model.recycle = pyo.Constraint(components, rule=recycle_rule)
    
    # define the objective function
    model.obj = pyo.Objective(
        expr = 100 * (2207*model.FP + 50*model.Fpurge - 168*model.FA - 252*model.FB - 2.22*model.Feff_sum 
                      - 84*model.FG - 60*model.V*rho)/(600 * model.V * rho),
        sense=pyo.maximize
    )

    return model

In [14]:
def batch_reactor_model(XA0, temp_profile):
    """
    Reformulates the variable temperature batch reactor model for parameter estimation

    Parameters
    ----------
    XA0: float,
        Initial mass fraction of species A
    temp_profile: Pandas series or list,
        Temperature profile from optimal experimental design

    Returns
    -------
    model: Pyomo model of the variable temperature batch reactor
    
    """
    model = pyo.ConcreteModel()

    # define sets
    reaction_number = [1, 2, 3]
    model.t = dae.ContinuousSet(bounds=[0, 3]) # hour

    # define the model parameters
    model.alpha_1 = pyo.Var(bounds=(0, None), initialize=10)
    model.alpha_2 = pyo.Var(bounds=(0, None), initialize=10)
    model.alpha_3 = pyo.Var(bounds=(0, None), initialize=10)
    model.E1 = pyo.Var(bounds=(0, None), initialize=50)
    model.E2 = pyo.Var(bounds=(0, None), initialize=50)
    model.E3 = pyo.Var(bounds=(0, None), initialize=50)
    
    # add the mass fraction variables
    model.XA = pyo.Var(model.t, bounds=(0, 1), initialize=XA0)
    model.XB = pyo.Var(model.t, bounds=(0, 1), initialize=1-XA0)
    model.XC = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XE = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XP = pyo.Var(model.t, bounds=(0, 1), initialize=0)
    model.XG = pyo.Var(model.t, bounds=(0, 1), initialize=0)

    # add the temperature variables
    model.T_reparam = pyo.Var(model.t, bounds=(0, 1))

    # add the rate constants
    model.k_reparam = pyo.Var(reaction_number, model.t, bounds=(0, None))
    model.k = pyo.Var(reaction_number, model.t, bounds=(0, None))
    
    # calculate the reparameterized rate constants
    def k_reparam_rule(m, i, t):
        if i == 1:
            return m.k_reparam[i, t] == m.alpha_1 - m.E1 * m.T_reparam[t]
        elif i == 2:
            return m.k_reparam[i, t] == m.alpha_2 - m.E2 * m.T_reparam[t]
        else:
            return m.k_reparam[i, t] == m.alpha_3 - m.E3 * m.T_reparam[t]
            
    model.k_reparam_eq = pyo.Constraint(
        reaction_number, model.t, rule=k_reparam_rule
    )
    
    # calculate the original rate constants
    def k_rule(m, i, t):
        return m.k[i, t] == pyo.exp(m.k_reparam[i, t])
    
    model.k_eq = pyo.Constraint(
        reaction_number, model.t, rule=k_rule
    )

    # add the differential equations for XA, XB, XC, XE, XP, and XG
    model.dXA = dae.DerivativeVar(model.XA, wrt=model.t)
    model.dXB = dae.DerivativeVar(model.XB, wrt=model.t)
    model.dXC = dae.DerivativeVar(model.XC, wrt=model.t)
    model.dXE = dae.DerivativeVar(model.XE, wrt=model.t)
    model.dXG = dae.DerivativeVar(model.XG, wrt=model.t)
    # model.dXP = dae.DerivativeVar(model.XP, wrt=model.t)

    @model.Constraint(model.t)
    def xa_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXA[t] == - m.k[1, t] * m.XA[t] * m.XB[t]
    
    @model.Constraint(model.t)
    def xb_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXB[t] == - (m.k[1, t] * m.XA[t] * m.XB[t] + m.k[2, t] * m.XB[t] * m.XC[t])
    
    @model.Constraint(model.t)
    def xc_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXC[t] == 2 * m.k[1, t] * m.XA[t] * m.XB[t] - 2 * m.k[2, t] * m.XB[t] * m.XC[t] - m.k[3, t] * m.XC[t] * m.XP[t]
    
    @model.Constraint(model.t)
    def xe_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXE[t] == 2 * m.k[2, t] * m.XB[t] * m.XC[t]
    
    @model.Constraint(model.t)
    def xg_rate_ode(m, t):
        if t == m.t.first():
            return pyo.Constraint.Skip
        return m.dXG[t] == 1.5 * m.k[3, t] * m.XC[t] * m.XP[t]

    # add the mass fraction constraint
    @model.Constraint(model.t)
    def sum_mass_fraction(m, t):
        return m.XA[t] + m.XB[t] + m.XC[t] + m.XE[t] + m.XG[t] + m.XP[t] == 1

    # fix the initial conditions
    t0 = model.t.first()
    model.XA_init = pyo.Constraint(expr=model.XA[t0] == XA0)
    model.XB_init = pyo.Constraint(expr=model.XB[t0] == 1 - model.XA[t0])
    model.XC_init = pyo.Constraint(expr=model.XC[t0] == 0.0)
    model.XE_init = pyo.Constraint(expr=model.XE[t0] == 0.0)
    model.XG_init = pyo.Constraint(expr=model.XG[t0] == 0.0)

    # discretize the model
    disc = pyo.TransformationFactory("dae.finite_difference")
    disc.apply_to(model, nfe=90, scheme="BACKWARD")

    # add the optimal temperature profile
    for indx, t in enumerate(time_vals):
        model.T_reparam[t].fix(temp_profile[indx])
    
    return model